In [ ]:
import requests            # library to send requests to websites / APIs
import pandas as pd         # tables (DataFrame)

print("requests version:", requests.__version__)

requests version: 2.32.4


In [ ]:
# The address (URL) of the API
url = "https://jsonplaceholder.typicode.com/posts"

# Send a GET request and store the reply in 'resp'
resp = requests.get(url, timeout=10)

# Always check for errors right after the request
resp.raise_for_status()

resp

<Response [200]>

👆 `<Response [200]>` means **success**. Let us look at more details.

In [ ]:
print("Status code :", resp.status_code)
print("Is it OK?   :", resp.ok)
print("Data format :", resp.headers["Content-Type"])

Status code : 200
Is it OK?   : True
Data format : application/json; charset=utf-8


In [ ]:
# First 300 characters of the raw text
resp.text[:300]

'[\n  {\n    "userId": 1,\n    "id": 1,\n    "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",\n    "body": "quia et suscipit\\nsuscipit recusandae consequuntur expedita et cum\\nreprehenderit molestiae ut ut quas totam\\nnostrum rerum est autem sunt rem eveniet architect'

In [ ]:
# .json() converts the JSON text into Python list/dict
data = resp.json()

print("Type of data     :", type(data))
print("Number of items  :", len(data))

Type of data     : <class 'list'>
Number of items  : 100


In [ ]:
# Look at the first item (a dictionary)
data[0]

{'userId': 1,
 'id': 1,
 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit',
 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}

In [ ]:
# Access one value from the dictionary
print(data[0]["title"])

sunt aut facere repellat provident occaecati excepturi optio reprehenderit


In [ ]:
posts = pd.DataFrame(data)

posts.head()

,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


In [ ]:
# Pick only the columns we want
posts_small = pd.DataFrame(data, columns=["userId", "id", "title"])

posts_small.head()

,userId,id,title
0,1,1,sunt aut facere repellat provident occaecati e...
1,1,2,qui est esse
2,1,3,ea molestias quasi exercitationem repellat qui...
3,1,4,eum et est occaecati
4,1,5,nesciunt quas odio


In [ ]:
# Now we can use all pandas tools! How many posts did each user write?
posts["userId"].value_counts().sort_index()

,count
userId,
1,10
2,10
3,10
4,10
5,10
6,10
7,10
8,10
9,10


In [ ]:
# Ask only for posts written by user 1
resp = requests.get("https://jsonplaceholder.typicode.com/posts",
                    params={"userId": 1}, timeout=10)
resp.raise_for_status()

print("Final URL used:", resp.url)       # see how requests built the URL
print("Posts received:", len(resp.json()))

Final URL used: https://jsonplaceholder.typicode.com/posts?userId=1
Posts received: 10


In [ ]:
user1_posts = pd.DataFrame(resp.json())
user1_posts[["id", "title"]].head()

,id,title
0,1,sunt aut facere repellat provident occaecati e...
1,2,qui est esse
2,3,ea molestias quasi exercitationem repellat qui...
3,4,eum et est occaecati
4,5,nesciunt quas odio


### C2. Another endpoint – comments
Different URL endings (called **endpoints**) give different data.

In [ ]:
comments = pd.DataFrame(requests.get("https://jsonplaceholder.typicode.com/comments",
                                     params={"postId": 1}, timeout=10).json())
comments[["id", "name", "email"]]

,id,name,email
0,1,id labore ex et quam laborum,Eliseo@gardner.biz
1,2,quo vero reiciendis velit similique earum,Jayne_Kuhic@sydney.com
2,3,odio adipisci rerum aut animi,Nikita@garfield.biz
3,4,alias odio sit,Lew@alysha.tv
4,5,vero eaque aliquid doloribus et culpa,Hayden@althea.biz


In [ ]:
url = "https://api.github.com/repos/pandas-dev/pandas/issues"

try:
    resp = requests.get(url, params={"per_page": 10}, timeout=10)   # per_page = how many items
    resp.raise_for_status()
    gh_data = resp.json()
    print("Live data received! Items:", len(gh_data))
except requests.exceptions.RequestException as e:
    print("Could not get live data:", e)
    print("Using sample data instead.")
    gh_data = [
        {"number": 1, "title": "BUG: sample issue one", "labels": [{"name": "Bug"}], "state": "open"},
        {"number": 2, "title": "ENH: sample issue two", "labels": [{"name": "Enhancement"}, {"name": "Docs"}], "state": "open"},
        {"number": 3, "title": "DOC: sample issue three", "labels": [], "state": "open"},
    ]

Live data received! Items: 10


In [ ]:
# Make a DataFrame with only the columns we need
issues = pd.DataFrame(gh_data, columns=["number", "title", "labels", "state"])
issues

,number,title,labels,state
0,72274,"ENH: add normalize, sort, ascending, bins, dro...","[{'id': 11527065895, 'node_id': 'LA_kwDOAA0YD8...",open
1,72273,BUG: avoid crash when plotting empty time-seri...,[],open
2,72272,CI: Update pixi.lock,"[{'id': 48070600, 'node_id': 'MDU6TGFiZWw0ODA3...",open
3,72271,PERF/BUG: skip rebuilding the join output in I...,"[{'id': 76811, 'node_id': 'MDU6TGFiZWw3NjgxMQ=...",open
4,72270,BUG: honor fill_value in PeriodIndex resample....,"[{'id': 11527065895, 'node_id': 'LA_kwDOAA0YD8...",open
5,72269,BUG preserve file order for pyarrow usecols,"[{'id': 11527065930, 'node_id': 'LA_kwDOAA0YD8...",open
6,72268,Fix for #72202: pandas nightly depreciated,"[{'id': 11527065930, 'node_id': 'LA_kwDOAA0YD8...",open
7,72267,PERF: wrap Arrow libjoin output directly in In...,"[{'id': 8935311, 'node_id': 'MDU6TGFiZWw4OTM1M...",open
8,72266,Fix read fwf comment 53922,"[{'id': 11527065930, 'node_id': 'LA_kwDOAA0YD8...",open
9,72265,BUG: Return NA for masked var/std when ddof >=...,"[{'id': 11527065930, 'node_id': 'LA_kwDOAA0YD8...",open


In [ ]:
resp = requests.get("https://jsonplaceholder.typicode.com/posts/99999", timeout=10)

print("Status code:", resp.status_code)   # 404 = Not Found

Status code: 404


`raise_for_status()` turns a bad status code into a Python error. We catch it with `try / except`.

In [ ]:
try:
    resp = requests.get("https://jsonplaceholder.typicode.com/posts/99999", timeout=10)
    resp.raise_for_status()                      # raises an error if status is 4xx or 5xx
    print(resp.json())
except requests.exceptions.HTTPError as e:
    print("HTTP error happened:", e)
except requests.exceptions.RequestException as e:
    print("Some other network problem:", e)

HTTP error happened: 404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/posts/99999


In [ ]:
def api_to_dataframe(url, params=None):
    """Fetch JSON (a list of dictionaries) from an API and return a DataFrame."""
    try:
        resp = requests.get(url, params=params, timeout=10)   # 1. send request
        resp.raise_for_status()                               # 2. check for errors
        return pd.DataFrame(resp.json())                      # 3. convert to DataFrame
    except requests.exceptions.RequestException as e:
        print("Error:", e)
        return pd.DataFrame()                                 # empty table if failed

# Test it with different endpoints
todos = api_to_dataframe("https://jsonplaceholder.typicode.com/todos", {"userId": 1})
todos.head()

,userId,id,title,completed
0,1,1,delectus aut autem,False
1,1,2,quis ut nam facilis et officia qui,False
2,1,3,fugiat veniam minus,False
3,1,4,et porro tempora,True
4,1,5,laboriosam mollitia et enim quasi adipisci qui...,False


In [ ]:
# How many todos are completed vs not completed for user 1?
todos["completed"].value_counts()

,count
completed,
True,11
False,9
